# Full Dataset Reference Analysis

This notebook uses the full SIIM-ISIC 2020 dataset for exploratory/reference
purposes.

The main group project uses a smaller subset, so results in this notebook
may differ from the main project results.

This notebook is provided as reference only and is not part of the main
training pipeline.

# CHECK PYTORCH

In [67]:
import sys
import torch
import pandas as pd
import numpy as np

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4090


# LET PYTHON INSPECT DATA

In [68]:
import os

raw_folder = r"E:\ADS2002_Melanoma\data\raw"
files = os.listdir(raw_folder)

print("Files and folders in raw folder:")
for file in files:
    print(file)

Files and folders in raw folder:
ISIC_2020_Training_Duplicates.csv
ISIC_2020_Training_GroundTruth.csv
ISIC_2020_Training_GroundTruth_v2.csv
train


In [69]:
image_folder = r"E:\ADS2002_Melanoma\data\raw\train"
image_files = os.listdir(image_folder)

print("number of images in train folder:", len(image_files))

print("First 5 image files:")
for file in image_files[:5]:
    print(file)

number of images in train folder: 33126
First 5 image files:
ISIC_0015719.jpg
ISIC_0052212.jpg
ISIC_0068279.jpg
ISIC_0074268.jpg
ISIC_0074311.jpg


## LOAD CSV FILES

In [81]:
ground_truth_file = r"E:\ADS2002_Melanoma\data\raw\ISIC_2020_Training_GroundTruth.csv"
metadata_file = r"E:\ADS2002_Melanoma\data\raw\ISIC_2020_Training_GroundTruth_v2.csv"
duplicate_files = r"E:\ADS2002_Melanoma\data\raw\ISIC_2020_Training_Duplicates.csv"

ground_truth_df = pd.read_csv(ground_truth_file)
metadata_df = pd.read_csv(metadata_file)
duplicate_df = pd.read_csv(duplicate_files)

In [82]:
print("Ground truth:")
display(ground_truth_df.head())

print("Metadata:")
display(metadata_df.head())

print("Duplicate files:")
display(duplicate_df.head())

Ground truth:


,image_name,patient_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target
0,ISIC_2637011,IP_7279968,male,45.0,head/neck,unknown,benign,0
1,ISIC_0015719,IP_3075186,female,45.0,upper extremity,unknown,benign,0
2,ISIC_0052212,IP_2842074,female,50.0,lower extremity,nevus,benign,0
3,ISIC_0068279,IP_6890425,female,45.0,head/neck,unknown,benign,0
4,ISIC_0074268,IP_8723313,female,55.0,upper extremity,unknown,benign,0


Metadata:


,image_name,patient_id,lesion_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target
0,ISIC_2637011,IP_7279968,IL_7972535,male,45.0,head/neck,unknown,benign,0
1,ISIC_0015719,IP_3075186,IL_4649854,female,45.0,upper extremity,unknown,benign,0
2,ISIC_0052212,IP_2842074,IL_9087444,female,50.0,lower extremity,nevus,benign,0
3,ISIC_0068279,IP_6890425,IL_4255399,female,45.0,head/neck,unknown,benign,0
4,ISIC_0074268,IP_8723313,IL_6898037,female,55.0,upper extremity,unknown,benign,0


Duplicate files:


,image_name_1,image_name_2
0,ISIC_0079038,ISIC_8521950
1,ISIC_0087297,ISIC_4755972
2,ISIC_0088137,ISIC_4201955
3,ISIC_0112097,ISIC_5934021
4,ISIC_0148783,ISIC_7460560


In [83]:
print("Ground truth dataframe shape:", ground_truth_df.shape)
print("Metadata dataframe shape:", metadata_df.shape)
print("Duplicate files dataframe shape:", duplicate_df.shape)

Ground truth dataframe shape: (33126, 8)
Metadata dataframe shape: (33126, 9)
Duplicate files dataframe shape: (425, 2)


In [84]:
print("Ground truth columns:")
print(ground_truth_df.columns.tolist())

print("Metadata columns:")
print(metadata_df.columns.tolist())

print("Duplicate files columns:")
print(duplicate_df.columns.tolist())

Ground truth columns:
['image_name', 'patient_id', 'sex', 'age_approx', 'anatom_site_general_challenge', 'diagnosis', 'benign_malignant', 'target']
Metadata columns:
['image_name', 'patient_id', 'lesion_id', 'sex', 'age_approx', 'anatom_site_general_challenge', 'diagnosis', 'benign_malignant', 'target']
Duplicate files columns:
['image_name_1', 'image_name_2']


In [85]:
metadata_without_lesion = metadata_df.drop(columns=['lesion_id'])
same_data = metadata_without_lesion.equals(ground_truth_df)
print("Do the ground truth and metadata (without lesion column) contain the same data?", same_data)

Do the ground truth and metadata (without lesion column) contain the same data? True


## FROM NOW, WORK ON metadata_df

In [92]:
df = metadata_df.copy()

print('Missing values in each column:')
print(df.isnull().sum())

print('\nNumber of benign and malignant cases:')
print(df['benign_malignant'].value_counts())

print('\nPercentage of benign and malignant cases:')
print(df['benign_malignant'].value_counts(normalize=True) * 100)

Missing values in each column:
image_name                         0
patient_id                         0
lesion_id                          0
sex                               65
age_approx                        68
anatom_site_general_challenge    527
diagnosis                          0
benign_malignant                   0
target                             0
dtype: int64

Number of benign and malignant cases:
benign_malignant
benign       32542
malignant      584
Name: count, dtype: int64

Percentage of benign and malignant cases:
benign_malignant
benign       98.237034
malignant     1.762966
Name: proportion, dtype: float64


In [93]:
print('\nNumber of images:', len(df))

print('\nNumber of unique patient_ids:', df['patient_id'].nunique())

print('\nNumber of unique lesion_ids:', df['lesion_id'].nunique())


Number of images: 33126

Number of unique patient_ids: 2056

Number of unique lesion_ids: 32701


In [94]:
first_duplicate = duplicate_df.iloc[0]

image_1 = first_duplicate['image_name_1']
image_2 = first_duplicate['image_name_2']

duplicate_example = df[
    df['image_name'].isin([image_1, image_2])
]

display(duplicate_example)

,image_name,patient_id,lesion_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target
17,ISIC_0079038,IP_5295861,IL_1642984,male,70.0,torso,unknown,benign,0
28155,ISIC_8521950,IP_5295861,IL_1642984,male,70.0,torso,unknown,benign,0


In [95]:
same_patient = 0
same_lesion = 0
same_target = 0

for i in range(len(duplicate_df)):
    image_1 = duplicate_df.iloc[i]['image_name_1']
    image_2 = duplicate_df.iloc[i]['image_name_2']
    
    row_1 = df[df['image_name'] == image_1].iloc[0]
    row_2 = df[df['image_name'] == image_2].iloc[0]
    
    if row_1['patient_id'] == row_2['patient_id']:
        same_patient += 1
    if row_1['lesion_id'] == row_2['lesion_id']:
        same_lesion += 1
    if row_1['target'] == row_2['target']:
        same_target += 1

print("Total number of duplicate pairs:", len(duplicate_df))
print(f"Number of duplicate pairs with the same patient_id: {same_patient}")
print(f"Number of duplicate pairs with the same lesion_id: {same_lesion}")
print(f"Number of duplicate pairs with the same target: {same_target}")

print("\nUnique image_name_1 values in duplicate_df:", duplicate_df['image_name_1'].nunique())
print("Unique image_name_2 values in duplicate_df:", duplicate_df['image_name_2'].nunique())

Total number of duplicate pairs: 425
Number of duplicate pairs with the same patient_id: 425
Number of duplicate pairs with the same lesion_id: 417
Number of duplicate pairs with the same target: 425

Unique image_name_1 values in duplicate_df: 425
Unique image_name_2 values in duplicate_df: 425


In [96]:
overlap_count = 0

for image in duplicate_df['image_name_1']:
    if image in duplicate_df['image_name_2'].values:
        overlap_count += 1

print(f"Number of overlapping image names between image_name_1 and image_name_2: {overlap_count}")

Number of overlapping image names between image_name_1 and image_name_2: 0


In [97]:
images_to_remove = []

for image in duplicate_df['image_name_2']:
    images_to_remove.append(image)

cleaned_df = df[~df['image_name'].isin(images_to_remove)].copy()

remaining_duplicates = 0

for image in images_to_remove:
    if image in cleaned_df['image_name'].values:
        remaining_duplicates += 1

print(f"Number of remaining duplicate pairs: {remaining_duplicates}")

Number of remaining duplicate pairs: 0


# EXPLORE CLEAN DATA

In [98]:
print('Missing values in each column:')
print(cleaned_df.isnull().sum())

print('\nNumber of benign and malignant cases:')
print(cleaned_df['benign_malignant'].value_counts())

print('\nPercentage of benign and malignant cases:')
print(cleaned_df['benign_malignant'].value_counts(normalize=True) * 100)

Missing values in each column:
image_name                         0
patient_id                         0
lesion_id                          0
sex                               41
age_approx                        44
anatom_site_general_challenge    526
diagnosis                          0
benign_malignant                   0
target                             0
dtype: int64

Number of benign and malignant cases:
benign_malignant
benign       32120
malignant      581
Name: count, dtype: int64

Percentage of benign and malignant cases:
benign_malignant
benign       98.223296
malignant     1.776704
Name: proportion, dtype: float64


In [99]:
# The model wants AGE, SEX, ANATOMICAL SITE
print('Sex counts:')
print(cleaned_df['sex'].value_counts(dropna=False))

print('\nAge summary:')
print(cleaned_df['age_approx'].describe())

print('\nAnatomical site counts:')
print(cleaned_df['anatom_site_general_challenge'].value_counts(dropna=False))

Sex counts:
sex
male      16869
female    15791
NaN          41
Name: count, dtype: int64

Age summary:
count    32657.000000
mean        48.814037
std         14.393214
min          0.000000
25%         40.000000
50%         50.000000
75%         60.000000
max         90.000000
Name: age_approx, dtype: float64

Anatomical site counts:
anatom_site_general_challenge
torso              16636
lower extremity     8319
upper extremity     4906
head/neck           1815
NaN                  526
palms/soles          375
oral/genital         124
Name: count, dtype: int64


In [100]:
patient_image_count = cleaned_df['patient_id'].value_counts()
print(patient_image_count.head())

print('\nNumber of patients:', cleaned_df['patient_id'].nunique())

print('\nImages per patient:')
print(patient_image_count.describe())

patient_id
IP_7279968    115
IP_4479736    115
IP_4938382    115
IP_4382720    115
IP_0656529    114
Name: count, dtype: int64

Number of patients: 2056

Images per patient:
count    2056.000000
mean       15.905156
std        15.323415
min         2.000000
25%         5.000000
50%        12.000000
75%        22.000000
max       115.000000
Name: count, dtype: float64


In [101]:
# How many patients actually have melanoma?
patient_ids = cleaned_df['patient_id'].unique()

patient_list = []

for patient in patient_ids:

    patient_rows = cleaned_df[
        cleaned_df['patient_id'] == patient
    ]

    number_of_images = len(patient_rows)

    has_melanoma = patient_rows['target'].max()

    patient_list.append([
        patient,
        number_of_images,
        has_melanoma
    ]) 

In [102]:
patient_df = pd.DataFrame(
    patient_list,
    columns=[
        'patient_id',
        'number_of_images',
        'has_melanoma'
    ]
)

display(patient_df.head())

,patient_id,number_of_images,has_melanoma
0,IP_7279968,115,0
1,IP_3075186,24,0
2,IP_2842074,5,0
3,IP_6890425,22,1
4,IP_8723313,20,0


In [103]:
print('Total patients:')
print(len(patient_df))

print('\nPatients with/without melanoma:')
print(patient_df['has_melanoma'].value_counts())

print('\nPercentage:')
print(patient_df['has_melanoma'].value_counts(normalize=True)*100)

Total patients:
2056

Patients with/without melanoma:
has_melanoma
0    1628
1     428
Name: count, dtype: int64

Percentage:
has_melanoma
0    79.182879
1    20.817121
Name: proportion, dtype: float64


# PATIENT-LEVEL TRAIN / VALIDATION / TEST SPLIT

In [104]:
from sklearn.model_selection import train_test_split

In [105]:
# First split: 70% train, 30% temporary
train_patients, temp_patients = train_test_split(
    patient_df,
    test_size = 0.30,
    random_state = 42,
    stratify = patient_df['has_melanoma']
)

In [106]:
# Split the remaining 30% in half
val_patients, test_patients = train_test_split(
    temp_patients,
    test_size = 0.50,
    random_state = 42,
    stratify = temp_patients['has_melanoma']
)

In [107]:
print('Training patients:', len(train_patients))
print('Validation patients:', len(val_patients))
print('Test patients:', len(test_patients))

Training patients: 1439
Validation patients: 308
Test patients: 309


In [108]:
print("Training:")
print(train_patients["has_melanoma"].value_counts())
print()

print("Validation:")
print(val_patients["has_melanoma"].value_counts())
print()

print("Test:")
print(test_patients["has_melanoma"].value_counts())

print("Training melanoma percentage:")
print(train_patients["has_melanoma"].mean() * 100)

print("\nValidation melanoma percentage:")
print(val_patients["has_melanoma"].mean() * 100)

print("\nTest melanoma percentage:")
print(test_patients["has_melanoma"].mean() * 100)

Training:
has_melanoma
0    1139
1     300
Name: count, dtype: int64

Validation:
has_melanoma
0    244
1     64
Name: count, dtype: int64

Test:
has_melanoma
0    245
1     64
Name: count, dtype: int64
Training melanoma percentage:
20.847810979847118

Validation melanoma percentage:
20.77922077922078

Test melanoma percentage:
20.711974110032365


In [109]:
# Get the patient IDs
train_patient_ids = train_patients["patient_id"].values
val_patient_ids = val_patients["patient_id"].values
test_patient_ids = test_patients["patient_id"].values

In [110]:
# Create the image-level datasets
train_df = cleaned_df[
    cleaned_df["patient_id"].isin(train_patient_ids)
].copy()

val_df = cleaned_df[
    cleaned_df["patient_id"].isin(val_patient_ids)
].copy()

test_df = cleaned_df[
    cleaned_df["patient_id"].isin(test_patient_ids)
].copy()

In [111]:
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

Training images: 23098
Validation images: 4657
Test images: 4946


In [112]:
# Verify zero patient leakage
train_ids = set(train_df["patient_id"])
val_ids = set(val_df["patient_id"])
test_ids = set(test_df["patient_id"])

In [113]:
train_val_overlap = train_ids.intersection(val_ids)
train_test_overlap = train_ids.intersection(test_ids)
val_test_overlap = val_ids.intersection(test_ids)

print("Patients shared between train and validation:", len(train_val_overlap))
print("Patients shared between train and test:", len(train_test_overlap))
print("Patients shared between validation and test:", len(val_test_overlap))

Patients shared between train and validation: 0
Patients shared between train and test: 0
Patients shared between validation and test: 0


In [114]:
# Check the malignant image distribution
print("Training:")
print(train_df["target"].value_counts())
print()

print("Validation:")
print(val_df["target"].value_counts())
print()

print("Test:")
print(test_df["target"].value_counts())

print("Training malignant image percentage:")
print(train_df["target"].mean() * 100)

print("\nValidation malignant image percentage:")
print(val_df["target"].mean() * 100)

print("\nTest malignant image percentage:")
print(test_df["target"].mean() * 100)

Training:
target
0    22702
1      396
Name: count, dtype: int64

Validation:
target
0    4568
1      89
Name: count, dtype: int64

Test:
target
0    4850
1      96
Name: count, dtype: int64
Training malignant image percentage:
1.7144341501428695

Validation malignant image percentage:
1.9111015675327463

Test malignant image percentage:
1.9409623938536191


In [125]:
# Save split files
processed_folder = r"E:\ADS2002_Melanoma\data\processed"
splits_folder = r"E:\ADS2002_Melanoma\data\splits"

os.makedirs(processed_folder, exist_ok = True)
os.makedirs(splits_folder, exist_ok = True)

In [126]:
cleaned_df.to_csv(
    r"E:\ADS2002_Melanoma\data\processed\cleaned_metadata.csv",
    index = False
)

In [127]:
train_df.to_csv(
    r"E:\ADS2002_Melanoma\data\splits\train.csv",
    index=False
)

val_df.to_csv(
    r"E:\ADS2002_Melanoma\data\splits\validation.csv",
    index=False
)

test_df.to_csv(
    r"E:\ADS2002_Melanoma\data\splits\test.csv",
    index=False
)

print("Files saved.")

Files saved.


In [128]:
print(os.listdir(splits_folder))

['test.csv', 'train.csv', 'validation.csv']
